# 08 · DataFrames: el siguiente nivel

**Caso:** una tienda tiene todas sus ventas en `pedidos.csv` (85 pedidos de 3 meses, con un `DatetimeIndex`). Tenemos también una tabla de `clientes.csv` y otra de `productos.csv`. Vamos a limpiarlas, combinarlas y analizarlas.

**Lo que vas a aprender:**

1. Lidiar con **NaN** (datos faltantes): detectarlos, quitarlos o rellenarlos.
2. **Concatenar** dataframes: apilar y pegar.
3. **Cruzar** tablas con `merge` (joins estilo SQL).
4. **Agrupar y resumir** con `groupby` + `agg`.
5. **Trabajar con fechas** como índice (`DatetimeIndex`).

---

## 0 · Lo que ya vimos

En la sesión anterior aprendiste a:

- Crear/leer DataFrames y explorar con `head`, `info`, `describe`.
- Seleccionar con `loc` / `iloc`.
- Filtrar con condiciones booleanas.

Lo que **no** vimos y vamos a ver hoy: qué hacer cuando faltan datos, cómo unir varias tablas, cómo resumir por grupos, y cómo manejar fechas como índice.

---

## Datos del caso

Las tablas viven en el repo público. Las cargamos desde la URL raw de GitHub.

**Detalle importante:** `pedidos.csv` tiene la fecha como columna normal. Al leerla le decimos a pandas que la convierta en **índice temporal** (`DatetimeIndex`) — esto nos permitirá filtrar por mes, por rango de fechas, etc.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = 'https://raw.githubusercontent.com/nodmintuf/notebooks-ml-sesiones/main/data'

URL_PEDIDOS   = f'{BASE}/pedidos.csv'
URL_CLIENTES  = f'{BASE}/clientes.csv'
URL_PRODUCTOS = f'{BASE}/productos.csv'

### Cargar las 3 tablas

In [ ]:
pedidos = pd.read_csv(URL_PEDIDOS, parse_dates=['fecha'], index_col='fecha')
clientes  = pd.read_csv(URL_CLIENTES)
productos = pd.read_csv(URL_PRODUCTOS)

print('pedidos  :', pedidos.shape, '· índice:', type(pedidos.index).__name__)
print('clientes :', clientes.shape)
print('productos:', productos.shape)

### Echamos un vistazo

In [ ]:
pedidos.head()

In [ ]:
clientes.head()

In [ ]:
productos

### Filtrar por fecha — la ventaja del `DatetimeIndex`

In [ ]:
# Selección por mes (string reconocible)
pedidos.loc['2025-02']

In [ ]:
# Rango de fechas con boolean indexing (funciona aunque no haya pedido ese día exacto)
inicio = '2025-02-10'
fin    = '2025-02-20'
pedidos.loc[(pedidos.index >= inicio) & (pedidos.index <= fin)]

In [ ]:
# Atajo: solo pedidos de febrero
pedidos.loc[pedidos.index.month == 2]

---

## 1 · NaN: el enemigo silencioso

**¿Por qué hay NaN?** Tres motivos típicos:

- **Error humano:** se olvidaron de teclear el precio, no se rellenó un campo.
- **Valor desconocido:** no sabemos quién hizo el pedido anónimo.
- **No aplica:** en enero y febrero no había campaña de descuentos.

NaN es "veneno" para los modelos: la mayoría de algoritmos fallan o lo ignoran en silencio. Tienes que decidir tú qué hacer.

### 1.1 · Detectar: `isna()`

In [ ]:
# ¿Dónde hay NaN en pedidos?
pedidos.isna()

In [ ]:
# ¿Cuántos NaN por columna?
pedidos.isna().sum()

In [ ]:
# ¿Hay alguna fila con NaN?
pedidos.isna().any(axis=1).sum(), 'filas con al menos un NaN'

### 1.2 · Quitar: `dropna()`

In [ ]:
# Por defecto, quita CUALQUIER fila que tenga AL MENOS un NaN
pedidos.dropna()

In [ ]:
# Solo si TODA la fila es NaN (raro, pero existe)
pedidos.dropna(how='all')

In [ ]:
# Quitar NaN solo de una columna concreta
pedidos.dropna(subset=['precio_unit'])

### 1.3 · Rellenar: `fillna()`

In [ ]:
# Con un valor constante
pedidos['precio_unit'].fillna(0)

In [ ]:
# Con la media de la columna
pedidos['precio_unit'].fillna(pedidos['precio_unit'].mean())

In [ ]:
# Con la mediana (más robusta si hay valores extremos)
pedidos['precio_unit'].fillna(pedidos['precio_unit'].median())

### 1.4 · Propagar valores: `ffill` y `bfill`

In [ ]:
# ffill: propaga el valor ANTERIOR hacia delante
# Útil cuando los datos están ordenados en el tiempo y el valor 'se mantiene'
pedidos['precio_unit'].ffill()

In [ ]:
# bfill: propaga el valor SIGUIENTE hacia atrás
pedidos['precio_unit'].bfill()

### 1.5 · ¿Cuándo usar cada estrategia?

| Situación | Estrategia |
|---|---|
| El NaN significa "0" (p. ej. un descuento que no se aplicó) | `fillna(0)` |
| Es un valor numérico y quieres mantener la media del conjunto | `fillna(df['col'].mean())` |
| Hay outliers que tiran de la media | `fillna(df['col'].median())` |
| Los datos están ordenados y el valor "se arrastra" (ej. precio del día anterior) | `ffill` |
| Tienes pocas filas y no puedes permitirte perderlas | `fillna` (no `dropna`) |
| La fila está demasiado incompleta para ser útil | `dropna` |

### Ejercicios 1 · NaN

1. ¿Cuántas filas de `pedidos` tienen `precio_unit` NaN?
2. Crea `pedidos_sin_precio_nan` quitando las filas con `precio_unit` NaN. ¿Cuántas filas te quedan?
3. Crea `pedidos_precio_limpio` rellenando los `precio_unit` NaN con la **mediana del producto** correspondiente (pista: `pedidos.groupby('producto')['precio_unit'].transform('median')`).
4. ¿Cuántos pedidos tienen `descuento` NaN? ¿Por qué?

In [ ]:
# Tu código aquí

---

## 2 · Concat: apilar y pegar

**Idea:** juntar varios dataframes, **uno encima de otro** (vertical) o **uno al lado del otro** (horizontal).

### 2.1 · Vertical: apilar filas
Por defecto `concat` apila hacia abajo (mismo esquema). Usamos DataFrames pequeños para ver el efecto claro.

In [ ]:
enero = pd.DataFrame({
    'pedido_id': [1, 2, 3],
    'fecha':     ['2025-01-15', '2025-01-22', '2025-01-30'],
    'producto':  ['A', 'B', 'A'],
    'cantidad':  [2, 1, 3],
})

febrero = pd.DataFrame({
    'pedido_id': [4, 5, 6],
    'fecha':     ['2025-02-05', '2025-02-12', '2025-02-18'],
    'producto':  ['C', 'B', 'A'],
    'cantidad':  [1, 4, 2],
})

# Apilar: las columnas se emparejan por nombre
pd.concat([enero, febrero])

### 2.2 · `ignore_index=True`
Resetea el índice para que vaya de 0 a N-1 sin repeticiones.

In [ ]:
df = pd.concat([enero, febrero], ignore_index=True)
df.index

### 2.3 · Horizontal: pegar columnas
Con `axis=1` pega dataframes **por filas** (mismo índice).

In [ ]:
# Truco: añade una columna con un prefijo y concatena horizontalmente
demo = pd.concat([enero, enero.add_prefix('dup_')], axis=1)
demo

### 2.4 · Esquemas diferentes

**Cuidado:** si los dataframes no tienen las mismas columnas, `concat` rellena con `NaN` donde falta.

In [ ]:
febrero_v2 = pd.DataFrame({
    'pedido_id': [4, 5, 6],
    'fecha':     ['2025-02-05', '2025-02-12', '2025-02-18'],
    'producto':  ['C', 'B', 'A'],
    'cantidad':  [1, 4, 2],
    'descuento': [0, 5, 10],   # columna extra
})

# 'enero' no tiene 'descuento' → se rellena con NaN
pd.concat([enero, febrero_v2], ignore_index=True)

### Ejercicios 2 · Concat

1. Filtra `pedidos` por mes para crear `ped_ene`, `ped_feb` y `ped_mar` (DataFrames separados).
2. Apila los 3 con `pd.concat` e `ignore_index=True`. ¿Cuántas filas tiene el resultado?
3. Ahora añade la columna `mes` (string) a cada DataFrame **antes** de concatenar. Compara con el resultado de hacer concat sin esa columna.

In [ ]:
# Tu código aquí

---

## 3 · Merge: cruzar tablas (joins estilo SQL)

**Idea:** juntar dos dataframes **por una columna común** (la "llave"). Es lo que en SQL es un `JOIN`.

Por ejemplo: `pedidos` tiene `cliente_id`. `clientes` tiene `cliente_id` + `nombre` + `ciudad`. Con merge, cada pedido se "enriquece" con los datos de su cliente.

### 3.1 · `merge` básico (inner join por defecto)
Sólo se quedan las filas que **existen en ambas tablas**.

In [ ]:
# inner: solo pedidos cuyo cliente_id está en clientes
m = pd.merge(pedidos, clientes, on='cliente_id', how='inner')
m.head()

In [ ]:
print('pedidos:', len(pedidos))
print('tras inner merge:', len(m))
print('(la diferencia son los pedidos con cliente_id NaN o no registrado)')

### 3.2 · Left join: mantener todos los de la izquierda
Aquí 'izquierda' = `pedidos`. Nos quedamos con **todos** los pedidos, y los que no tienen cliente rellenan con NaN.

In [ ]:
m = pd.merge(pedidos, clientes, on='cliente_id', how='left')
m.head()

In [ ]:
# Fíjate: los pedidos con cliente_id NaN o 126/127/128 (no están en clientes)
# se quedan con NaN en nombre, ciudad, segmento
m[m['nombre'].isna()].head()

### 3.3 · Right y Outer join
- **Right:** mantiene todos los de la derecha (todos los clientes), aunque no tengan pedidos.
- **Outer:** mantiene los de ambas tablas.

In [ ]:
# right: TODOS los clientes aparecen, aunque no tengan pedidos
m = pd.merge(pedidos, clientes, on='cliente_id', how='right')
m[m['pedido_id'].isna()].head()

### 3.4 · Columnas con nombre distinto: `left_on` / `right_on`
Si las columnas llave no se llaman igual en ambas tablas.

In [ ]:
# Ejemplo: si en 'clientes' la llave fuera 'id' en vez de 'cliente_id'
clientes_renombrado = clientes.rename(columns={'cliente_id': 'id'})
m = pd.merge(pedidos, clientes_renombrado, left_on='cliente_id', right_on='id')
m.head()

### 3.5 · Múltiples llaves y sufijos
Si hay columnas con el mismo nombre en ambas tablas, pandas añade sufijos.

In [ ]:
# Ejemplo: si por error las dos tablas tuvieran una columna 'nombre'
left  = pd.DataFrame({'id': [1, 2], 'nombre': ['A', 'B'], 'valor': [10, 20]})
right = pd.DataFrame({'id': [1, 2], 'nombre': ['X', 'Y'], 'valor': [100, 200]})
pd.merge(left, right, on='id', suffixes=('_izq', '_der'))

### Ejercicios 3 · Merge

1. Haz un **left join** de `pedidos` con `clientes` por `cliente_id`. ¿Cuántos pedidos quedan con `nombre` NaN?
2. Haz un **left join** de `pedidos` con `productos` por `producto`. ¿Tienen todos los pedidos un producto en la tabla?
3. ¿Cuántos pedidos hay de clientes de **Madrid**? (Pista: tras el merge, filtra por `ciudad == 'Madrid'`.)
4. ¿Y de clientes con `segmento == 'premium'`?

In [ ]:
# Tu código aquí

---

## 4 · GroupBy: dividir y resumir

**Idea:** "split-apply-combine".

1. **Split:** divides las filas en grupos según una columna (o el índice).
2. **Apply:** aplicas una operación a cada grupo (suma, media, conteo...).
3. **Combine:** juntas los resultados en una tabla resumen.

### 4.1 · `groupby` + `size`
¿Cuántas filas hay por grupo?

In [ ]:
# Por mes, aprovechando el DatetimeIndex
pedidos.groupby(pedidos.index.month).size()

In [ ]:
# Equivalente, pero con el nombre del mes en vez del número
pedidos.groupby(pedidos.index.month_name()).size()

### 4.2 · `groupby` + `agg` con una métrica

In [ ]:
# Primero añadimos una columna 'importe' provisional para los ejemplos
# (NaN en precio_unit → 0 para no romper la suma)
base = pedidos.assign(
    importe=pedidos['cantidad'] * pedidos['precio_unit'].fillna(0)
)

# Total facturado por mes
base.groupby(base.index.month)['importe'].sum()

### 4.3 · Múltiples métricas a la vez

In [ ]:
# Por mes: suma, media y conteo
base.groupby(base.index.month)['importe'].agg(['sum', 'mean', 'count'])

### 4.4 · Varias columnas a la vez

In [ ]:
# Por cada producto, total facturado y nº de pedidos
resumen_prod = (base
    .groupby('producto')
    .agg(importe_total=('importe', 'sum'),
         n_pedidos=('pedido_id', 'count'),
         unidades=('cantidad', 'sum'))
    .sort_values('importe_total', ascending=False))
resumen_prod

### 4.5 · Agrupar por varias columnas

In [ ]:
# Por mes Y producto
resumen_mes_prod = (base
    .groupby([base.index.month, 'producto'])['importe']
    .sum()
    .unstack()
    .fillna(0)
    .round(2))
resumen_mes_prod

### 4.6 · Resample: series temporales
Como el índice es temporal, podemos usar `resample` (como `groupby` pero para fechas).

In [ ]:
# Total facturado por mes, usando resample
base['importe'].resample('ME').sum()

### Ejercicios 4 · GroupBy

1. ¿Cuántos pedidos hay por **categoría** de producto? (Necesitarás mergear con `productos`.)
2. ¿Cuál es el **ticket medio** (importe medio por pedido) por **ciudad** del cliente?
3. ¿Qué **cliente** ha hecho más pedidos en estos 3 meses? Muestra los 5 primeros.
4. ¿Cuál fue el día con mayor importe facturado? (Pista: `groupby(pedidos.index.date)`.)

In [ ]:
# Tu código aquí

---

## 5 · Mini-práctica integradora

Tienes `pedidos`, `clientes` y `productos`. Demuestra que dominas las 4 herramientas resolviendo este caso de principio a fin. **Trabaja en la celda de abajo** (puedes crear variables intermedias, hacer merge temporal, etc.).

**Pasos:**

1. **Limpia** los `precio_unit` NaN usando la **mediana del producto** correspondiente.
2. **Calcula** la columna `importe = cantidad * precio_unit` (post-limpieza).
3. **Rellena** los `descuento` NaN con 0 (en enero/febrero no había descuentos, pero我们需要 un valor para operar).
4. **Enriquece** `pedidos` haciendo merge con `clientes` (left, por `cliente_id`) y con `productos` (left, por `producto`).
5. **Responde:**
   a. Importe total por **mes y categoría** (pista: `.unstack()`).
   b. Top 3 **ciudades** por importe total, con su ticket medio.
   c. ¿Qué **segmento** de cliente (premium / standard) factura más?
6. **Bonus:** ¿cuánto se dejó de facturar por los descuentos? (Pista: `descuento > 0`.)

In [ ]:
# Tu código aquí